# Thota Mitra: Pilot Validation Notebook
## Status: Pilot validation pipeline built; awaiting multi-season historical diary dataset from partner plantation.
## Historical Weather Evaluation & Decision Engine Protocol

> ### ⚠️ CRITICAL INTEGRITY NOTE (SEVA 2026 DEMO)
> **Status: Pilot validation with one plantation planned. CLAIM NO ACCURACY.**
> No published numeric thresholds or clinical accuracy rates exist for the heuristic weights (35/30/20/15).
> Real multi-season farm diary outbreak records from field growers are needed before any precision, recall, or accuracy metrics can be reported to judges.
> This notebook establishes the transparent, reproducible pipeline to download historical Open-Meteo weather and evaluate it against real farm records when collected.

In [ ]:
import json
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# 1. Load Heuristic Risk Rules
with open('../config/risk_rules.json', 'r', encoding='utf-8') as f:
    rules = json.load(f)

print("Validation Status:", rules['_meta']['status'])
print("Distinction Note:", rules['_meta']['agronomic_distinction']['software_numeric_heuristics'])

In [ ]:
# 2. Connect to Open-Meteo Historical Archive API
# Plantation Coordinates: Thirthahalli, Shivamogga district (13.6934° N, 75.2427° E)
lat, lon = 13.6934, 75.2427
start_date = "2025-06-01"
end_date = "2025-08-31"

archive_url = (
    f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}"
    f"&start_date={start_date}&end_date={end_date}"
    f"&daily=precipitation_sum,relative_humidity_2m_mean,temperature_2m_max,temperature_2m_min,wind_speed_10m_max"
    f"&timezone=auto"
)

print(f"Querying Open-Meteo Archive: {archive_url}")
try:
    req = urllib.request.Request(archive_url, headers={'User-Agent': 'ThotaMitra-Pilot-Validation'})
    with urllib.request.urlopen(req) as resp:
        raw_data = json.loads(resp.read().decode('utf-8'))
    df_weather = pd.DataFrame(raw_data['daily'])
    df_weather['date'] = pd.to_datetime(df_weather['time'])
    print(f"Successfully pulled {len(df_weather)} days of real historical weather from Open-Meteo Archive.")
except Exception as err:
    print(f"Archive API unavailable or network restricted: {err}")
    df_weather = None

if df_weather is not None:
    display(df_weather.head())

In [ ]:
# 3. Compute Conducive-Conditions Index for Each Date
def calculate_index(row, prev_days):
    # Consecutive rainy days (IMD standard >= 2.5mm)
    streak = 0
    for p in reversed(prev_days):
        if p['precipitation_sum'] >= 2.5:
            streak += 1
        else:
            break
    if row['precipitation_sum'] >= 2.5:
        streak += 1
        
    rain_subscore = 100 if streak >= 4 else (65 if streak >= 2 else (35 if streak == 1 else 10))
    rh = row['relative_humidity_2m_mean']
    rh_subscore = 100 if rh >= 90 else (75 if rh >= 82 else 30)
    temp = (row['temperature_2m_max'] + row['temperature_2m_min']) / 2
    temp_subscore = 100 if (20 <= temp <= 28) else 30
    
    # Heuristic composite
    return round((rain_subscore * 0.40) + (rh_subscore * 0.40) + (temp_subscore * 0.20))

if df_weather is not None:
    indices = []
    for i in range(len(df_weather)):
        prev = df_weather.iloc[max(0, i-4):i].to_dict('records')
        curr = df_weather.iloc[i].to_dict()
        indices.append(calculate_index(curr, prev))
    df_weather['conduciveness_index'] = indices
    print("Conduciveness Index calculated across monsoon days.")

In [ ]:
# 4. Load Farm Diary Entries from SQLite
import sqlite3

try:
    conn = sqlite3.connect('../thota_mitra.db')
    df_diary = pd.read_sql_query("SELECT * FROM farm_diary_entries", conn)
    conn.close()
    print(f"Loaded {len(df_diary)} recorded diary entries from plantation database.")
    display(df_diary)
except Exception as e:
    print(f"Could not query database directly: {e}")
    df_diary = pd.DataFrame()

In [ ]:
# 5. Honest Validation Protocol Statement
print("=" * 60)
print("VALIDATION PROTOCOL FOR SEVA 2026 PRESENTATION")
print("=" * 60)
print("Current status: Pilot validation with one plantation planned.")
print("Accuracy claimed: NONE (Heuristic decision support only).")
print("Reason: Reliable precision/recall requires at least 2 full monsoon")
print("seasons of recorded outbreak logs and microplot controls across")
print("multiple growers in the taluk.")
print("-" * 60)
print("Next Step: Export farm diary entries as growers log Koleroga events")
print("during the upcoming monsoon to calibrate weights against field observations.")